# 05 · 1 m DEM pilot (Experiment F)

1. **F1 inventory** of `data/raw/dem_1m/tnm_urls.json` (put the JSON there on Drive): parsing, duplicates / overlaps, study-area intersection, storage estimate, header check of a few tiles over HTTP.
2. **F2 balanced pilot**: quota tiles (short faults missed by G1, systems missed by the CNN, high disagreement), control tiles without faults, stratified random tiles. Selection never looks at DEM content.
3. **Approve the download** explicitly (`runtime.dem_download = True`; same experiment id) after reading the estimate.
4. **F3** block-wise 1 m morphology -> exact 100 m aggregation -> `data/processed/dem_1m_100m/dem_features_100m.tif` (registered as `dem:*` features).
5. **F4** controlled benchmarks on identical DEM-covered pixels: `f4_g1_dem` (LightGBM) and `f4_dual_dem` (gated dual encoder).

In [ ]:
# --- session bootstrap (idempotent; identical in notebooks 00-07) ---------------------------------
REPO = 'OWNER/REPOSITORY'
COMMIT = 'main'
DRIVE_ROOT = '/content/drive/MyDrive/GEMS_2'
import os, subprocess, sys
from google.colab import drive
drive.mount('/content/drive')
token = ''
try:
    from google.colab import userdata
    token = userdata.get('GITHUB_TOKEN') or ''
except Exception:
    pass
url = f'https://{token}@github.com/{REPO}.git' if token else f'https://github.com/{REPO}.git'
if not os.path.exists('/content/GEMS/.git'):
    subprocess.run(['git', 'clone', '-q', url, '/content/GEMS'], check=True)
subprocess.run(['git', '-C', '/content/GEMS', 'fetch', '-q', 'origin'], check=True)
subprocess.run(['git', '-C', '/content/GEMS', 'checkout', '-q', COMMIT], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '/content/GEMS[colab]'], check=True)
if '/content/GEMS' not in sys.path:
    sys.path.insert(0, '/content/GEMS')
from gems_bench.runtime import colab
colab.configure(drive_root=DRIVE_ROOT)
colab.stage_core(extra=('data/raw/dem_1m/tnm_urls.json', 'data/processed/dem_1m_100m')); colab.stage_legacy();

In [ ]:
import json
from gems_bench.experiment import Experiment, ExperimentConfig
C = '/content/GEMS/configs/experiments/'
f1 = Experiment.from_yaml(C + 'f1_dem_pilot.yaml')
f1.prepare()
print(json.loads((f1.rd.reports / 'dem_storage_estimate.json').read_text()))
if f1.rd.has_table('dem_corner_check'):
    display(f1.rd.load_table('dem_corner_check'))
f1.rd.load_table('dem_pilot_tiles')[['tile_id', 'reason', 'region', 'label_density', 'short_missed_px', 'cnn_missed_px']]

In [ ]:
# Explicit approval: download the selected tiles (resumable) and process them (resumable per tile)
APPROVE_DOWNLOAD = False
if APPROVE_DOWNLOAD:
    cfg = ExperimentConfig.from_yaml(C + 'f1_dem_pilot.yaml')
    cfg.raw.setdefault('runtime', {})['dem_download'] = True      # non-semantic: same experiment id
    f1 = Experiment(cfg)
    f1.train()
    f1.evaluate()
    display(f1.rd.load_table('dem_coverage_report'))

In [ ]:
# F4: controlled benchmarks on identical DEM-covered pixels (refresh the feature store so dem:* bands exist)
for n in ('f4_g1_dem', 'f4_dual_dem'):
    e = Experiment.from_yaml(C + n + '.yaml')
    e.run_all()
    display(e.rd.load_table('dem_coverage_report'))
    if e.rd.has_table('promotion_decisions'):
        display(e.rd.load_table('promotion_decisions'))